In [ ]:
# ============================================================
# Capstone Project - Neural Networks
# Slogan Generator and Industry Classifier
# ============================================================

# ---------------------------
# 1. Imports
# ---------------------------
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# ---------------------------
# 2. Reproducibility
# ---------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 3. Load dataset
# ============================================================
# IMPORTANT:
# Replace the file name below with the exact slogan dataset
# file provided in your task folder if it differs.
DATA_PATH = "slogans.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

# ============================================================
# 4. Select relevant columns and handle missing values
# ============================================================
# Adjust these names if your dataset uses slightly different labels.
# This block tries common alternatives.
possible_slogan_cols = ["slogan", "Slogan", "tagline", "Tagline"]
possible_industry_cols = ["industry", "Industry", "category", "Category"]
possible_business_cols = ["business_name", "Business Name", "company", "Company", "name", "Name"]

def find_column(possible_names, columns):
    for col in possible_names:
        if col in columns:
            return col
    return None

slogan_col = find_column(possible_slogan_cols, df.columns)
industry_col = find_column(possible_industry_cols, df.columns)
business_col = find_column(possible_business_cols, df.columns)

if slogan_col is None or industry_col is None:
    raise ValueError(
        "Could not find the slogan and industry columns. "
        "Please check the dataset column names."
    )

if business_col is None:
    df["business_name_fallback"] = ""
    business_col = "business_name_fallback"

df = df[[business_col, slogan_col, industry_col]].copy()
df.columns = ["business_name", "slogan", "industry"]

df.dropna(subset=["slogan", "industry"], inplace=True)
df["business_name"] = df["business_name"].fillna("")
df["slogan"] = df["slogan"].astype(str).str.strip()
df["business_name"] = df["business_name"].astype(str).str.strip()
df["industry"] = df["industry"].astype(str).str.strip()

df = df[(df["slogan"] != "") & (df["industry"] != "")].reset_index(drop=True)

print("\nCleaned dataset shape:", df.shape)
print(df.head())

# ============================================================
# 5. Text preprocessing
# ============================================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_slogan"] = df["slogan"].apply(clean_text)
df["clean_business_name"] = df["business_name"].apply(clean_text)
df["combined_text"] = (df["clean_business_name"] + " " + df["clean_slogan"]).str.strip()

print("\nSample cleaned text:")
print(df[["business_name", "slogan", "industry", "combined_text"]].head())

# ============================================================
# 6. Tokenisation
# ============================================================
def tokenize(text):
    return text.split()

df["slogan_tokens"] = df["clean_slogan"].apply(tokenize)
df["combined_tokens"] = df["combined_text"].apply(tokenize)

# Add start/end tokens for generation
df["generator_tokens"] = df.apply(
    lambda row: ["<industry_" + row["industry"].replace(" ", "_").lower() + ">"]
    + row["slogan_tokens"]
    + ["<eos>"],
    axis=1
)

print("\nSample tokenised slogan:")
print(df["generator_tokens"].iloc[0])

# ============================================================
# 7. Build vocabularies
# ============================================================
special_tokens = ["<pad>", "<unk>", "<eos>"]

all_gen_tokens = []
for seq in df["generator_tokens"]:
    all_gen_tokens.extend(seq)

gen_counter = Counter(all_gen_tokens)
gen_vocab = special_tokens + sorted([tok for tok in gen_counter if tok not in special_tokens])
gen_word2idx = {word: idx for idx, word in enumerate(gen_vocab)}
gen_idx2word = {idx: word for word, idx in gen_word2idx.items()}

print("\nGenerator vocab size:", len(gen_vocab))

all_cls_tokens = []
for seq in df["combined_tokens"]:
    all_cls_tokens.extend(seq)

cls_vocab = ["<pad>", "<unk>"] + sorted(set(all_cls_tokens))
cls_word2idx = {word: idx for idx, word in enumerate(cls_vocab)}
cls_idx2word = {idx: word for word, idx in cls_word2idx.items()}

print("Classifier vocab size:", len(cls_vocab))

# ============================================================
# 8. Encode industries
# ============================================================
label_encoder = LabelEncoder()
df["industry_label"] = label_encoder.fit_transform(df["industry"])

num_classes = len(label_encoder.classes_)
print("\nIndustries:", list(label_encoder.classes_))
print("Number of industries:", num_classes)

# ============================================================
# 9. Prepare generator training data
# ============================================================
generator_sequences = []

for token_list in df["generator_tokens"]:
    encoded = [gen_word2idx.get(tok, gen_word2idx["<unk>"]) for tok in token_list]
    for i in range(1, len(encoded)):
        input_seq = encoded[:i]
        target_word = encoded[i]
        generator_sequences.append((input_seq, target_word))

print("\nNumber of generator training samples:", len(generator_sequences))

# ============================================================
# 10. Prepare classifier training data
# ============================================================
MAX_LEN = max(len(tokens) for tokens in df["combined_tokens"])
print("Maximum classifier sequence length:", MAX_LEN)

def encode_classifier_text(tokens, vocab, max_len):
    encoded = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    if len(encoded) < max_len:
        encoded += [vocab["<pad>"]] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return encoded

X_classifier = np.array([
    encode_classifier_text(tokens, cls_word2idx, MAX_LEN)
    for tokens in df["combined_tokens"]
], dtype=np.int64)

y_classifier = df["industry_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_classifier,
    y_classifier,
    test_size=0.2,
    random_state=SEED,
    stratify=y_classifier
)

X_train_tensor = torch.tensor(X_train, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.long).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

print("\nClassifier training shape:", X_train_tensor.shape)
print("Classifier testing shape:", X_test_tensor.shape)

# ============================================================
# 11. Pad generator sequences
# ============================================================
GEN_MAX_LEN = max(len(seq[0]) for seq in generator_sequences)

def pad_generator_sequence(seq, max_len, pad_idx):
    if len(seq) < max_len:
        seq = seq + [pad_idx] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

X_gen = np.array([
    pad_generator_sequence(seq, GEN_MAX_LEN, gen_word2idx["<pad>"])
    for seq, target in generator_sequences
], dtype=np.int64)

y_gen = np.array([target for _, target in generator_sequences], dtype=np.int64)

X_gen_tensor = torch.tensor(X_gen, dtype=torch.long).to(device)
y_gen_tensor = torch.tensor(y_gen, dtype=torch.long).to(device)

print("\nGenerator input shape:", X_gen_tensor.shape)
print("Generator target shape:", y_gen_tensor.shape)

# ============================================================
# 12. Define slogan generator model
# ============================================================
class SloganGeneratorLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SloganGeneratorLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=gen_word2idx["<pad>"])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        last_output = output[:, -1, :]
        out = self.fc(last_output)
        return out

# ============================================================
# 13. Define slogan classifier model
# ============================================================
class IndustryClassifierLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(IndustryClassifierLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=cls_word2idx["<pad>"])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden[-1])
        return out

# ============================================================
# 14. Instantiate models
# ============================================================
GEN_EMBED_DIM = 64
GEN_HIDDEN_DIM = 128

CLS_EMBED_DIM = 64
CLS_HIDDEN_DIM = 128

generator_model = SloganGeneratorLSTM(
    vocab_size=len(gen_vocab),
    embedding_dim=GEN_EMBED_DIM,
    hidden_dim=GEN_HIDDEN_DIM,
    output_dim=len(gen_vocab)
).to(device)

classifier_model = IndustryClassifierLSTM(
    vocab_size=len(cls_vocab),
    embedding_dim=CLS_EMBED_DIM,
    hidden_dim=CLS_HIDDEN_DIM,
    output_dim=num_classes
).to(device)

print("\nGenerator model:")
print(generator_model)
print("\nClassifier model:")
print(classifier_model)

# ============================================================
# 15. Training helpers
# ============================================================
def train_generator(model, X, y, epochs=15, batch_size=64, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset_size = X.size(0)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        permutation = torch.randperm(dataset_size, device=device)

        for i in range(0, dataset_size, batch_size):
            indices = permutation[i:i + batch_size]
            batch_x = X[indices]
            batch_y = y[indices]

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Generator Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.4f}")

def train_classifier(model, X, y, epochs=15, batch_size=64, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset_size = X.size(0)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        permutation = torch.randperm(dataset_size, device=device)

        for i in range(0, dataset_size, batch_size):
            indices = permutation[i:i + batch_size]
            batch_x = X[indices]
            batch_y = y[indices]

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Classifier Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.4f}")

# ============================================================
# 16. Train both models
# ============================================================
train_generator(generator_model, X_gen_tensor, y_gen_tensor, epochs=15, batch_size=64, lr=0.001)
train_classifier(classifier_model, X_train_tensor, y_train_tensor, epochs=15, batch_size=64, lr=0.001)

# ============================================================
# 17. Evaluate classifier
# ============================================================
classifier_model.eval()
with torch.no_grad():
    test_outputs = classifier_model(X_test_tensor)
    test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()

test_accuracy = accuracy_score(y_test, test_preds)

print("\nClassifier Accuracy:", round(test_accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=label_encoder.classes_))

# ============================================================
# 18. Slogan generation function
# ============================================================
def generate_slogan(industry, model, max_words=12):
    model.eval()

    industry_token = "<industry_" + industry.replace(" ", "_").lower() + ">"
    if industry_token not in gen_word2idx:
        return f"Industry '{industry}' was not seen during training."

    generated_tokens = [industry_token]

    for _ in range(max_words):
        encoded = [gen_word2idx.get(tok, gen_word2idx["<unk>"]) for tok in generated_tokens]
        padded = pad_generator_sequence(encoded, GEN_MAX_LEN, gen_word2idx["<pad>"])
        input_tensor = torch.tensor([padded], dtype=torch.long).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1).cpu().numpy().flatten()

        next_idx = np.random.choice(len(probs), p=probs)
        next_word = gen_idx2word[next_idx]

        if next_word == "<eos>":
            break

        if next_word not in ["<pad>", "<unk>"]:
            generated_tokens.append(next_word)

    final_tokens = [tok for tok in generated_tokens[1:] if not tok.startswith("<industry_>")]
    return " ".join(final_tokens).strip()

# ============================================================
# 19. Classify a slogan function
# ============================================================
def classify_slogan(text, model):
    model.eval()

    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    encoded = encode_classifier_text(tokens, cls_word2idx, MAX_LEN)

    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()

    return label_encoder.inverse_transform([pred_idx])[0]

# ============================================================
# 20. Test slogan generation
# ============================================================
print("\nGenerated slogans by industry:")
for industry in label_encoder.classes_[:5]:
    slogan = generate_slogan(industry, generator_model, max_words=12)
    print(f"Industry: {industry}")
    print(f"Generated slogan: {slogan}")
    print("-" * 60)

# ============================================================
# 21. Combine both models
# ============================================================
print("\nGenerator + Classifier Combined Evaluation:")

comparison_rows = []

for industry in label_encoder.classes_[:10]:
    generated = generate_slogan(industry, generator_model, max_words=12)
    predicted_industry = classify_slogan(generated, classifier_model)

    comparison_rows.append({
        "Input Industry": industry,
        "Generated Slogan": generated,
        "Predicted Industry": predicted_industry,
        "Match": industry == predicted_industry
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df)

match_rate = comparison_df["Match"].mean()
print("\nMatch rate between intended and predicted industry:", round(match_rate, 4))

# ============================================================
# 22. Comments on differences
# ============================================================
print("\nObservations:")
print(
    "If the classifier predicts a different industry from the generator input, "
    "this may indicate that the generated slogan contains words or patterns that "
    "are more strongly associated with another industry in the training data. "
    "It may also show that the generator is producing generic slogans that do "
    "not clearly distinguish one industry from another."
)
